# Решения: venv, requirements и README

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
import json
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

_DATA_URL = "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_10_churn_logreg/data/bank_marketing_slim.csv"
DATA_PATH = next(
    (p for p in (
        Path("bank_marketing_slim.csv"),
        Path("../../data/bank_marketing_slim.csv"),
        Path("../data/bank_marketing_slim.csv"),
    ) if p.exists()),
    _DATA_URL,
)
CLI_SCRIPT = Path("../04_practice_metrics_cli/train_cli.py")
assert DATA_PATH.exists() and CLI_SCRIPT.exists()


## Урок. 1. Команды среды

In [ ]:
venv_commands = ['python -m venv .venv', '.venv\\Scripts\\activate', 'python -m pip install --upgrade pip', 'python -m pip install -r requirements.txt']
assert len(venv_commands) == 4


## Урок. 2–4. Зависимости

In [ ]:
required_packages = ['numpy', 'pandas', 'scikit-learn']
def package_line(distribution_name):
    try:
        return f"{distribution_name}=={version(distribution_name)}"
    except PackageNotFoundError as exc:
        raise RuntimeError(f"Пакет {distribution_name} не установлен") from exc

requirements_lines = [package_line(name) for name in required_packages]
requirements_text = '\n'.join(requirements_lines) + '\n'
assert requirements_text.count('\n') == 3


## Урок. 5. Фактические метрики

In [ ]:
cmd = [sys.executable, str(CLI_SCRIPT), '--data', str(DATA_PATH), '--threshold', '0.45']
proc = subprocess.run(cmd, capture_output=True, text=True, check=False)
assert proc.returncode == 0, proc.stderr
metrics = json.loads(proc.stdout)
assert metrics['duration_in_features'] is False


## Урок. 6. Каркас README

In [ ]:
readme_sections = ['Цель', 'Данные', 'Leakage', 'Установка', 'Запуск', 'Результаты', 'Ограничения']
assert len(readme_sections) == 7


## Урок. 7. README

In [ ]:
experiment_readme = f'''# Эксперимент: отклик на депозит

## Цель
Ранжировать клиентов банковской кампании по вероятности отклика `y=yes` и применить порог решения.

## Данные
Использован учебный `bank_marketing_slim.csv`, подготовленный из UCI Bank Marketing.

## Leakage
`duration` известна только после звонка и запрещена как признак. Скрипт проверяет `assert "duration" not in FEATURE_COLUMNS`, а JSON возвращает `duration_in_features=false`.

## Установка
```text
python -m venv .venv
.venv\\Scripts\\activate
python -m pip install -r requirements.txt
```

## Запуск
```text
python train_cli.py --data bank_marketing_slim.csv --threshold 0.45 --seed 63
```

## Результаты
- threshold: {metrics["threshold"]:.2f}
- accuracy: {metrics["accuracy"]:.3f}
- precision: {metrics["precision"]:.3f}
- recall: {metrics["recall"]:.3f}
- f1: {metrics["f1"]:.3f}

## Ограничения
Это учебный slim-срез, а не свежая выборка банка. Метрики относятся к одному стратифицированному split. Порог нужно пересчитать под реальный бюджет, период данных и цену ошибок.
'''
assert len(experiment_readme) >= 700 and f"{metrics['f1']:.3f}" in experiment_readme


## Урок. 8. Gate сдачи

In [ ]:
submission_files = {'train_cli.py', 'requirements.txt', 'README.md', 'metrics.json'}
required_files = {'train_cli.py', 'requirements.txt', 'README.md'}
missing = required_files - submission_files
assert missing == set()


## Урок. 9. Передача

In [ ]:
HANDOFF_NOTE = ("На чистой машине создать venv, активировать его и установить requirements.txt, затем запустить указанную CLI-команду. "
"Сверить JSON-ключи и убедиться, что duration_in_features=false: duration недоступна до звонка. "
"Полученные метрики должны совпасть при том же seed и версиях. Ограничение: slim-срез учебный, поэтому результат не является оценкой будущей банковской кампании.")
assert len(HANDOFF_NOTE) >= 260


## ДЗ. A1. Дерево

In [ ]:
tree_text = 'submission/\n  train_cli.py  # CLI модели\n  requirements.txt  # версии зависимостей\n  README.md  # инструкция и результаты\n  metrics.json  # сохранённый вывод, опционально\n'
assert len(tree_text) >= 120


## ДЗ. A2. Чек-лист

In [ ]:
readme_checks = {'goal': '## Цель' in experiment_readme, 'data': '## Данные' in experiment_readme, 'leakage': 'duration' in experiment_readme, 'install': 'python -m venv .venv' in experiment_readme, 'run': 'train_cli.py' in experiment_readme, 'metrics': 'f1:' in experiment_readme, 'limitations': '## Ограничения' in experiment_readme}
assert set(readme_checks.values()) == {True}


## ДЗ. A3. Воспроизведение

In [ ]:
reproduction_commands = ['python -m venv .venv', '.venv\\Scripts\\activate', 'python -m pip install --upgrade pip', 'python -m pip install -r requirements.txt', 'python train_cli.py --data bank_marketing_slim.csv --threshold 0.45 --seed 63']
assert len(reproduction_commands) == 5


## ДЗ. Challenge. Аудит README

In [ ]:
def audit_readme(text):
    return {'goal': '## Цель' in text, 'data': '## Данные' in text, 'leakage': 'duration' in text.lower(), 'install': 'python -m venv .venv' in text, 'run': 'train_cli.py' in text, 'metrics': all(name in text for name in ('precision', 'recall', 'f1')), 'limitations': '## Ограничения' in text}

audit = audit_readme(experiment_readme)
assert set(audit.values()) == {True}


## ДЗ. Challenge. Postmortem

In [ ]:
POSTMORTEM = ("Воспроизводимость ломается, если не зафиксированы версии библиотек: solver или кодировка могут измениться. "
"Другой seed меняет train/test и метрики; неверный относительный путь мешает найти CSV. Самый опасный тихий сбой — добавить duration и получить завышенное качество с недоступным признаком. "
"Даже при исправном запуске slim-срез не представляет будущую кампанию: нужны свежие данные, мониторинг доли yes и повторный выбор порога. Поэтому README фиксирует среду, команду, seed, leakage-guard и границы вывода.")
assert len(POSTMORTEM) >= 320
